In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv("D:\\forecasting\\data\\train.csv")
features = pd.read_csv("D:\\forecasting\\data\\features.csv")
stores = pd.read_csv("D:\\forecasting\\data\\stores.csv")

df = train.merge(
    features,
    on=["Store","Date","IsHoliday"],
    how="left"
)

df = df.merge(
    stores,
    on="Store",
    how="left"
)

df["Date"] = pd.to_datetime(df["Date"])

# Missing values
markdown_cols = [
    "MarkDown1",
    "MarkDown2",
    "MarkDown3",
    "MarkDown4",
    "MarkDown5"
]

df[markdown_cols] = df[markdown_cols].fillna(0)

df["CPI"] = df["CPI"].fillna(df["CPI"].median())
df["Unemployment"] = df["Unemployment"].fillna(
    df["Unemployment"].median()
)

In [2]:
df["Year"] = df["Date"].dt.year

df["Month"] = df["Date"].dt.month

df["Quarter"] = df["Date"].dt.quarter

df["Week"] = (
    df["Date"]
      .dt
      .isocalendar()
      .week
      .astype(int)
)

In [3]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["Type"] = le.fit_transform(df["Type"])

In [4]:
df = df.sort_values(
    ["Store","Dept","Date"]
)

df["lag_1"] = (
    df.groupby(["Store","Dept"])
      ["Weekly_Sales"]
      .shift(1)
)

df["lag_2"] = (
    df.groupby(["Store","Dept"])
      ["Weekly_Sales"]
      .shift(2)
)

df["lag_4"] = (
    df.groupby(["Store","Dept"])
      ["Weekly_Sales"]
      .shift(4)
)

df["lag_8"] = (
    df.groupby(["Store","Dept"])
      ["Weekly_Sales"]
      .shift(8)
)

In [5]:
df[
[
"Weekly_Sales",
"lag_1",
"lag_2",
"lag_4"
]
].head(15)

,Weekly_Sales,lag_1,lag_2,lag_4
0,24924.50,NaN,NaN,NaN
1,46039.49,24924.50,NaN,NaN
2,41595.55,46039.49,24924.50,NaN
3,19403.54,41595.55,46039.49,NaN
4,21827.90,19403.54,41595.55,24924.50
5,21043.39,21827.90,19403.54,46039.49
6,22136.64,21043.39,21827.90,41595.55
7,26229.21,22136.64,21043.39,19403.54
8,57258.43,26229.21,22136.64,21827.90
9,42960.91,57258.43,26229.21,21043.39


In [6]:
df["rolling_mean_4"] = (
    df.groupby(["Store", "Dept"])["Weekly_Sales"]
      .transform(lambda x: x.shift(1).rolling(4).mean())
)

df["rolling_std_4"] = (
    df.groupby(["Store", "Dept"])["Weekly_Sales"]
      .transform(lambda x: x.shift(1).rolling(4).std())
)

df["rolling_mean_8"] = (
    df.groupby(["Store", "Dept"])["Weekly_Sales"]
      .transform(lambda x: x.shift(1).rolling(8).mean())
)

In [7]:
df[
[
"Weekly_Sales",
"lag_1",
"rolling_mean_4",
"rolling_std_4"
]
].head(15)

,Weekly_Sales,lag_1,rolling_mean_4,rolling_std_4
0,24924.50,NaN,NaN,NaN
1,46039.49,24924.50,NaN,NaN
2,41595.55,46039.49,NaN,NaN
3,19403.54,41595.55,NaN,NaN
4,21827.90,19403.54,32990.7700,12832.106391
5,21043.39,21827.90,32216.6200,13554.047185
6,22136.64,21043.39,25967.5950,10467.484020
7,26229.21,22136.64,21102.8675,1222.784968
8,57258.43,26229.21,22809.2850,2325.929203
9,42960.91,57258.43,31666.9175,17206.391261


In [8]:
model_df = df.dropna().copy()

print(model_df.shape)

(395604, 27)


In [9]:
model_df.isnull().sum().sum()

np.int64(0)

In [11]:
model_df.to_csv(
    "D:\\forecasting\\data\\feature_engineered_data.csv",
    index=False
)